In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

df_text = pd.read_csv("Data/dataset_main.csv")

# same subset you cluster on downstream — keeps uid alignment simple
subset = df_text[(df_text['split'] == 'train') & (df_text['label'] == 1)].copy()

# fit on this subset's text; word + char n-grams, as discussed
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9)
X_tfidf = tfidf.fit_transform(subset['text'])

svd = TruncatedSVD(n_components=10, random_state=42)  # pick via cumvar, same as your PCA step
X_reduced = svd.fit_transform(X_tfidf)

out = pd.DataFrame(X_reduced, columns=[f"svd_{i}" for i in range(X_reduced.shape[1])])
out.insert(0, 'uid', subset['uid'].values)
out.to_csv("Data/features_tfidf_clustering.csv", index=False)